# Open-Weight Fine-Tuning: QLoRA → GGUF

**Author: A Taylor**

Fine-tune an open-weight model (Llama 3.3 70B / Qwen2.5 72B) so it reliably emits this stack's tool calls, then export it to GGUF and serve the same agent against it.

> Heavy deps live in `requirements-finetune.txt` and need a GPU host:
> `pip install -r requirements-finetune.txt`

## 1. Build agentic SFT data

Each example walks a scenario through parallel tool calls, tool results, and a grounded final answer — teaching the tool-calling "dialect".

In [ ]:
import sys
sys.path.insert(0, "..")

from src.training_data import build_example, run

# Inspect one example
example = build_example({
    "instrument": "Spectrometer",
    "material_name": "Indium Phosphide",
    "environment_location": "Jovian System",
    "thermal_effect": "Spectral Drift",
    "strategy_type": "Hybrid",
})
for msg in example["messages"]:
    print(msg["role"], "->", msg.get("content") or msg.get("tool_calls"))

# Build the full train/validation JSONL
run(output_dir="../data/finetune")

## 2. QLoRA fine-tune

4-bit base + LoRA adapters. Base model and hyperparameters come from `config/finetune_config.yaml`.

In [ ]:
from src.finetune import run_finetune

adapter_dir = run_finetune("../config/finetune_config.yaml")
print("Adapter saved to", adapter_dir)

## 3. Merge + quantize to GGUF

Requires a built llama.cpp checkout (path set in the config).

In [ ]:
from src.quantize import run as quantize_run

gguf_path = quantize_run("../config/finetune_config.yaml")
print("Quantized GGUF:", gguf_path)

## 4. Serve and run the agent against it

Serve the GGUF behind an OpenAI-compatible endpoint, then either set `provider: local` in `config/agent_config.yaml` and call `ThermalAgent.from_config()`, or wire the backend directly:

In [ ]:
from src.agent import ThermalAgent
from src.backends import LocalToolBackend
from src.tools import ToolDispatcher

backend = LocalToolBackend(model="thermal-agent", base_url="http://localhost:8000/v1")
agent = ThermalAgent(dispatcher=ToolDispatcher(), client=backend)

result = agent.run(
    "Instrument: Spectrometer\n"
    "Material: Indium Phosphide\n"
    "Environment: Jovian System\n"
    "Thermal Effect: Spectral Drift\n"
    "What thermal mitigation strategy should be used and why?"
)
print(result["answer"])